## Supervised Fine-Tuning (SFT) with Serverless customization on SageMaker AI

<div style="border: 2px solid #ff9900; border-radius: 8px; padding: 15px; background-color: #fff3e0; margin-bottom: 10px;">
<strong>⚠️ Compatibility Notice:</strong> This Immersion Day has been tested using the following SageMaker Distribution images:

<ul>
<li><strong>SageMaker Distribution Image 4.4.1</strong></li>
</ul>  
and the following SageMaker Python SDK version
<ul>
    <li><strong>SageMaker Python SDK version 3.13.1</strong></li>
</ul>
</div>

# Lab 1 – Prepare the Dataset

This is the first of three labs that fine-tune and deploy a large language model using **serverless model customization** on SageMaker AI — no training cluster to manage. The labs build on each other:

| Lab | Goal |
|---|---|
| **1 – Prepare data** (this notebook) | Curate, reformat, split, and register a medical-reasoning dataset. |
| **2 – Fine-tune** | Run a serverless SFT (LoRA) job on the registered data. |
| **3 – Deploy** | Host the fine-tuned model on a real-time endpoint and test it. |

**Why data preparation matters:** in supervised fine-tuning the model learns *purely from the examples you give it*. The format, quality, and split of this dataset directly determine what the model learns, so getting this step right is the foundation for everything that follows. For background on the customization workflow see [Customizing models with Amazon SageMaker AI](https://docs.aws.amazon.com/sagemaker/latest/dg/customize-model.html).

**In this notebook we:**

1. Stream a public medical reasoning dataset from the Hugging Face Hub.
2. Reformat each example into a `prompt`/`completion` pair with explicit `<think>` reasoning traces.
3. Split it into **train / validation / test** sets.
4. Upload the splits to S3 and **register them in the SageMaker AI Registry** so Lab 2 can reference them by name.

***

### Prerequisites

### Step 1 – Install requirements

Install the Python dependencies used in this notebook (Hugging Face `datasets`, `pandas`, `scikit-learn`, the SageMaker SDK, etc.). If you already ran the repo's top-level [`Setup.ipynb`](../../Setup.ipynb) these may already be present.

In [ ]:
# --- Lab dependencies (managed via uv) ---------------------------------------
# Installs THIS lab's complete, self-contained kernel dependencies from the
# lab requirements.txt using uv. Idempotent and fast when already satisfied.
# This is the only dependency step the lab needs - Setup.ipynb is not required.
import sys
!pip install -q uv
!uv pip install -q --python {sys.executable} -r requirements.txt

### Step 2 – Set up the SageMaker session

Establish the SageMaker [`Session`](https://sagemaker.readthedocs.io/en/stable/api/utility/session.html), resolve the **execution role**, and select the **default S3 bucket** where we will stage the prepared datasets.

In [ ]:
import boto3
from sagemaker.core.helper.session_helper import Session, get_execution_role

sess = Session()
sagemaker_session_bucket = None

if sagemaker_session_bucket is None and sess is not None:
    # set to default bucket if a bucket name is not given
    sagemaker_session_bucket = sess.default_bucket()

try:
    role = get_execution_role()
except ValueError:
    iam = boto3.client("iam")
    role = iam.get_role(RoleName="sagemaker_execution_role")["Role"]["Arn"]

s3_client = boto3.client("s3")
sess = Session(default_bucket=sagemaker_session_bucket)
bucket_name = sess.default_bucket()
default_prefix = sess.default_bucket_prefix

print(f"sagemaker role arn: {role}")
print(f"sagemaker bucket: {sess.default_bucket()}")
print(f"sagemaker session region: {sess.boto_region_name}")

***

### Step 3 – Prepare the dataset

We use [`FreedomIntelligence/medical-o1-reasoning-SFT`](https://huggingface.co/datasets/FreedomIntelligence/medical-o1-reasoning-SFT), an open dataset of medical questions paired with a chain-of-thought explanation and a final answer.

We **stream** the data (`streaming=True`) and `take(1500)` examples rather than downloading the whole dataset — streaming avoids materializing a large file locally, and 1,500 shuffled examples are plenty for a workshop-scale fine-tune. The `shuffle(buffer_size=500)` randomizes ordering so the splits are representative.

In [ ]:
import datasets
from datasets import load_dataset

dataset = (
    load_dataset(
        "FreedomIntelligence/medical-o1-reasoning-SFT",
        "en",
        split="train",
        streaming=True,
    )
    .take(1500)
    .shuffle(buffer_size=500)
)

dataset = datasets.Dataset.from_generator(lambda: dataset, features=dataset.features)

Load the streamed examples into a pandas DataFrame and preview the raw schema. Each row has a `Question`, a `Complex_CoT` (the reasoning trace), and a `Response` (the final answer) — we will reshape these next.

In [ ]:
import pandas as pd

df = pd.DataFrame(dataset)

df.head()

#### Split into train / validation / test

We carve the data into three disjoint sets, producing roughly a **70 / 10 / 20** split:

- **Train** – what the model actually learns from.
- **Validation** – held out *during* training so SageMaker can monitor for overfitting on data the model isn't learning from.
- **Test** – never seen during training; reserved for evaluating the final model.

Using a fixed `random_state` makes the split reproducible across runs.

In [ ]:
from sklearn.model_selection import train_test_split

train, val = train_test_split(df, test_size=0.2, random_state=42)
train, test = train_test_split(train, test_size=0.125, random_state=42)

print("Number of train elements: ", len(train))
print("Number of validation elements: ", len(val))
print("Number of test elements: ", len(test))

#### Reformat into the SFT schema

SageMaker SFT expects each training example as a **`prompt`** (the input) and a **`completion`** (the target the model should learn to produce). We map the raw columns accordingly and wrap the reasoning trace in explicit `<think> … </think>` tags followed by the final answer.

**Why the `<think>` tags?** They teach the model an explicit, separable *reasoning-then-answer* structure. At inference time you can show or hide the reasoning block, and the consistent delimiter makes the model's chain of thought easy to parse — you'll see this same block appear when we test the deployed model in Lab 3.

Note the **test** set uses `query`/`response` keys instead of `prompt`/`completion`, since it is for evaluation rather than training.

In [ ]:
from datasets import Dataset
from tqdm import tqdm


def prepare_dataset_train_val(sample):
    yield {
        "prompt": sample["Question"],
        "completion": f"<think>\n{sample['Complex_CoT']}\n</think>\n\n {sample['Response']}",
    }


def prepare_dataset_test(sample):
    yield {
        "query": sample["Question"],
        "response": f"<think>\n{sample['Complex_CoT']}\n</think>\n\n {sample['Response']}",
    }

Helper functions that apply the reformatting above to every row and return a Hugging Face `Dataset`.

In [ ]:
def convert_to_messages_train_val(dataset):
    """Iteratively run conversion on multi-turn conversation and flatten to messages"""
    records = []

    print("Original length: ", len(dataset))

    # Unroll your generator for every dataset row
    for row in tqdm(dataset, total=len(dataset), desc="Converting to messages"):
        for example in prepare_dataset_train_val(row):
            records.append(example)

    # Convert list of dicts → Hugging Face Dataset and return
    return Dataset.from_list(records)


def convert_to_messages_test(dataset):
    """Iteratively run conversion on multi-turn conversation and flatten to messages"""
    records = []

    print("Original length: ", len(dataset))

    # Unroll your generator for every dataset row
    for row in tqdm(dataset, total=len(dataset), desc="Converting to messages"):
        for example in prepare_dataset_test(row):
            records.append(example)

    # Convert list of dicts → Hugging Face Dataset and return
    return Dataset.from_list(records)

Run the conversion on all three splits and print one randomly chosen train and test example so you can eyeball the final `prompt`/`completion` (and `query`/`response`) format before uploading.

In [ ]:
from datasets import Dataset, DatasetDict
import json
from random import randint

train_dataset = Dataset.from_pandas(train)
val_dataset = Dataset.from_pandas(val)
test_dataset = Dataset.from_pandas(test)

dataset = DatasetDict(
    {"train": train_dataset, "val": val_dataset, "test": test_dataset}
)

train_dataset = convert_to_messages_train_val(dataset["train"])

print(json.dumps(train_dataset[randint(0, len(train_dataset) - 1)], indent=2))

val_dataset = convert_to_messages_train_val(dataset["val"])

test_dataset = convert_to_messages_test(dataset["test"])

print(json.dumps(test_dataset[randint(0, len(test_dataset) - 1)], indent=2))

### Step 4 – Upload the splits to Amazon S3

Serverless customization reads its inputs from S3, so we write each split to JSON Lines (`.jsonl`, one example per line) and upload it to the session's default bucket. The local copies are removed afterwards to keep the notebook environment clean.

In [ ]:
import shutil

In [ ]:
# save train_dataset to s3 using our SageMaker session
if default_prefix:
    input_path = f"{default_prefix}/datasets/serverless-model-customization-sft"
else:
    input_path = f"datasets/serverless-model-customization-sft"

train_dataset_s3_path = f"s3://{bucket_name}/{input_path}/train/dataset.jsonl"
val_dataset_s3_path = f"s3://{bucket_name}/{input_path}/val/dataset.jsonl"
test_dataset_s3_path = f"s3://{bucket_name}/{input_path}/test/dataset.jsonl"

In [ ]:
train_dataset.to_json("./data/train/dataset.jsonl", orient="records")
val_dataset.to_json("./data/val/dataset.jsonl", orient="records")
test_dataset.to_json("./data/test/dataset.jsonl", orient="records")

s3_client.upload_file(
    "./data/train/dataset.jsonl", bucket_name, f"{input_path}/train/dataset.jsonl"
)
s3_client.upload_file(
    "./data/val/dataset.jsonl", bucket_name, f"{input_path}/val/dataset.jsonl"
)
s3_client.upload_file(
    "./data/test/dataset.jsonl", bucket_name, f"{input_path}/test/dataset.jsonl"
)

shutil.rmtree("./data")

print(f"Training data uploaded to:")
print(train_dataset_s3_path)
print(val_dataset_s3_path)
print(test_dataset_s3_path)

### Step 5 – Register the datasets in the SageMaker AI Registry

Finally we register the train, validation, and test splits as **`DataSet`** assets in the SageMaker AI Registry. Registering data (alongside models and evaluators) gives you versioning and automatic **lineage** — SageMaker records which dataset produced which model — and lets Lab 2 reference each split **by name** instead of hard-coding S3 paths. See [Tracking and managing assets used in AI development with Amazon SageMaker AI](https://aws.amazon.com/blogs/machine-learning/tracking-and-managing-assets-used-in-ai-development-with-amazon-sagemaker-ai/).

The `customization_technique=CustomizationTechnique.SFT` tag marks the train/validation sets as SFT data; the test set is registered without it because it is for evaluation.

In [ ]:
from sagemaker.ai_registry.dataset import DataSet
from sagemaker.ai_registry.dataset_utils import CustomizationTechnique

In [ ]:
dataset_train = DataSet.create(
    name="medical-o1-reasoning-sft-train",
    source=train_dataset_s3_path,
    customization_technique=CustomizationTechnique.SFT,
    wait=True,
)

print(f"TRAINING_DATASET ARN: {dataset_train.arn}")

dataset_val = DataSet.create(
    name="medical-o1-reasoning-sft-val",
    source=val_dataset_s3_path,
    customization_technique=CustomizationTechnique.SFT,
    wait=True,
)

print(f"VALIDATION_DATASET ARN: {dataset_val.arn}")

dataset_test = DataSet.create(
    name="medical-o1-reasoning-sft-test",
    source=test_dataset_s3_path,
    wait=True,
)

print(f"TEST_DATASET ARN: {dataset_test.arn}")